# Mathematical Correctness Proof `core_logistic_regression_lasso.py`

**The goal:** Find the weights $w$ that minimize the total cost:
$$J(w) = L(w) + \alpha \|w\|_1$$
where $L(w)$ is the weighted logistic loss (how wrong the predictions are) and $\alpha \|w\|_1$ is the L1-regularization (a penalty that keeps weights small and pushes unimportant ones to exactly zero).

We prove that `fit` (the training function of `LassoLogisticRegression`) correctly executes **Proximal Gradient Descent** on $J(w)$. This means: the model should fit well ($L$ small) but not become too complex ($\|w\|_1$ small).

**The sigmoid function** converts any real number into a probability between 0 and 1:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
It is used to turn the raw weighted sum of features into a predicted probability $\hat{p}_i$ for each observation.

----------

## Loop Invariant (I)

At the beginning of the $t$-th iteration the following holds: After iteration $t$, the code must have the same number in self.coef_ that is also mathematically computed as $w^{(t)}$.

**Intuition:** Each iteration has two phases. First, take a step in the direction that reduces prediction error (gradient step). Second, shrink weights toward zero to enforce sparsity (proximal step). Together, these two steps define one correct iteration of Proximal Gradient Descent.

The **Proximal Operator** for L1 is the soft-thresholding function $S$: it takes a weight and shrinks it toward zero by a fixed amount. Weights that are already close to zero get set to exactly zero. This is what creates sparse solutions (feature selection).

Gradient Descent: First we perform an update based only on the smooth part of the cost function, which is the Weighted Logistic Loss $L(w)$.
Then we take the current weights $w^{(k-1)}$ and move them in the negative direction of the gradient $\nabla L$.
This gives us $z^{(k)}$, an intermediate weight vector. It minimizes the prediction error but does not yet account for the $L_1$-regularization (sparsity).

Since the $L_1$ penalty is not differentiable at zero, we cannot use a standard gradient. Instead, we use the Proximal Operator (for $L_1$ -> Soft-Thresholding Operator $S$).
This operator is applied to the intermediate vector $z^{(k)}$. It "shrinks" the weights toward zero based on the threshold $\eta \alpha$.
- If $|z_i| \le \eta \alpha$  => $w_i$ becomes exactly 0.
- If $|z_i| > \eta \alpha$ => $w_i$ is reduced by the value of the threshold.

> `self.coef_` $= w^{(t)}$ is the result of exactly $t$ correct Proximal Gradient steps, for all $k=1, \ldots, t$:
> $$z^{(k)} = w^{(k-1)} - \eta \, \nabla L(w^{(k-1)})$$
> $$w^{(k)} = S_{\eta\alpha}(z^{(k)})$$

We now need to show that the loop invariant holds for every step $t$.

----------

## IA - Initialization ($t = 0$)

**Intuition:** Before the loop starts, all weights are set to zero. This is the defined mathematical starting point.

To show: The invariant holds before the first iteration.

The code:
```python
self.coef_      = np.zeros(n_features)   # w^(0) = 0
self.intercept_ = 0.0                    # b^(0) = 0
```

Since self.coef_ is initialized as a zero vector, it is identical to the mathematical starting point $w^{(0)}$ at $t = 0$. The statement is therefore true. 

---------

## IS - Maintenance ($t -> t+1$)

**Intuition:** Each step updates the weights in exactly the way the math prescribes: first a gradient step (reduce error), then a proximal step (enforce sparsity). We show that after each iteration, the code matches the mathematical update.

**IH:** `self.coef_` $= w^{(t)}$ is the result of $t$ correct steps. We therefore assume that the invariant holds for $t$ as a hypothesis.

To show: After executing the loop body, `self.coef_` contains the value $w^{(t+1)}$ (See invariant). We must therefore show that the invariant also holds after $t+1$.

The update consists of two sub-steps:

**IS)a) Gradient**:
We need to show that coef_new in the code corresponds to the mathematical intermediate state $z^{(t+1)}$

```python
p_hat    = self._sigmoid(X @ self.coef_ + self.intercept_)   # sigmoid converts to probability
residual = p_hat - y
grad_coef = (X.T @ (sw * residual)) / sw_sum
coef_new  = self.coef_ - self.learning_rate * grad_coef
```
1. The sigmoid function converts the raw prediction $X w^{(t)} + b$ into a probability $\hat{p}_i = \sigma(x_i^\top w^{(t)} + b) \in (0,1)$.
2. The code computes the gradient of the weighted logistic loss: $\nabla L(w^{(t)}) =\frac{X^\top (sw \odot (\hat{p} - y))}{\sum sw}$.
3. It then updates the weights: coef_new $= w^{(t)} - \eta \nabla L(w^{(t)})$.
4. Mathematically, this defines the intermediate proposal $z^{(t+1)}$.

=> TRUE (coef_new $= z^{(t+1)}$)

**IS)b) Proximal mapping**:

**Intuition:** The soft-thresholding operator takes the intermediate weights and shrinks them toward zero. Weights below the threshold become exactly zero (feature selection). This is the step that enforces sparsity.

```python
coef_new = self._soft_threshold(coef_new, self.alpha * self.learning_rate)
```
1. The soft-thresholding operator $S_\lambda(z) = \operatorname{sign}(z) \cdot \max(|z| - \lambda, 0)$ is the proximal operator for the $L_1$-norm, solving: $$S_\lambda(z) = \arg\min_w \frac{1}{2}\|w - z\|^2 + \lambda\|w\|_1$$
2. Parameter mapping: with $\lambda = \eta\alpha$ (where $\eta$ is learning_rate and $\alpha$ is alpha), the code applies this operator to the intermediate state coef_new ($z^{(t+1)}$).
3. The code sets the final weights for the iteration: $$\text{self.coef\_} = S_{\eta\alpha}(z^{(t+1)}) = w^{(t+1)}$$

=> TRUE (self.coef_ $= w^{(t+1)}$)

Since $L(w)$ is convex and the gradient does not change too abruptly (it is bounded by the data), for sufficiently small `learning_rate`:
$$J(w^{(t+1)}) \leq J(w^{(t)})$$
i.e. each iteration approaches the global minimum. Subsequently `self.coef_` $= w^{(t+1)}$ is set, and the invariant holds for $t+1$. (TRUE)

--------

## Termination

**Intuition:** The algorithm stops in one of two ways. Either the weights stop changing meaningfully (convergence), or a hard limit on iterations is reached. In both cases the result is valid.

**Case 1 - Convergence:**
Stop when the biggest weight update falls below the threshold:
$$\delta = \|w^{(t+1)} - w^{(t)}\|_\infty < 10^{-4}$$
```python
if delta < self.tol:
    break
```
At this point the change is minimal and we are at a minimum that corresponds to the global minimum of $J(w)$. (TRUE)

**Case 2 - Maximum Iterations:**
Hard stop after 1,500 iterations, with a `ConvergenceWarning`.
```python
for iteration in range(self.max_iter):
```
According to the invariant, $w^{(T)}$ is the result of $T$ correct steps, however the convergence described in Case 1 is not fulfilled.

--------

## Conclusion

Since the invariant holds at the beginning (IA) and is maintained after each step (IS), `fit` correctly computes Proximal Gradient Descent on $J(w)$. The loop terminates in every case, and the result `self.coef_` corresponds to the global minimum due to the convexity of $J(w)$.